In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd
import random
import json
import os

# 봇 탐지를 피하기 위해 user agent로 설정
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:138.0) Gecko/20100101 Firefox/138.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36',
]

options = Options()
options.add_argument(f'user-agent={random.choice(user_agents)}')
options.add_experimental_option('excludeSwitches', ['enable-automation'])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument('--disable-blink-features=AutomationControlled')

# Chrome 드라이버 자동 다운로드 및 설정
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# navigator.webdriver 플래그 제거
driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
    'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
})

# 변수 정의
query = 'SK텔레콤'
start_date = '2025.04.01'  # CSV 파일명용
end_date = '2025.04.30'    # CSV 파일명용

# url_수집.ipynb에서 저장한 월별 링크 파일 경로
links_file_name = f"링크_{query}_{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}.json"
links_path = os.path.join(os.getcwd(), 'data', links_file_name)

In [11]:
# 링크 파일 로드 및 개수 확인
if os.path.exists(links_path):
    with open(links_path, 'r', encoding='utf-8') as f:
        naver_news_links = json.load(f)
    est_min = len(naver_news_links) * 1.85 / 60  # 평균 딜레이 기준 예상 소요 시간
    print(f'링크 {len(naver_news_links)}개 불러옴')
    print(f'예상 소요 시간: 약 {est_min:.0f}분')
else:
    raise FileNotFoundError(f'링크 파일 없음 — 날짜루프ver.ipynb를 먼저 실행하세요\n경로: {links_path}')

링크 5351개 불러옴
예상 소요 시간: 약 165분


In [ ]:
# 추출한 링크에 직접 방문하여 크롤링 진행 - 교수님 제공 코드 이용

# 크롤링 속도를 향상시키기 위해 time.sleep대신 WebDriverWait으로 변경 (다만 네트워크 불량 시 오래 걸릴 수 있음)
# WebDriverWait 이용 시 탐색 속도가 너무 빨라 봇 탐지 가능성 존재 -> time.sleep 이용 요망

# 진행상황 확인코드 추가

# 중간저장 파일 경로
checkpoint_name = f"체크포인트_{query}_{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}.json"
checkpoint_path = os.path.join(os.getcwd(), 'data', checkpoint_name)

# 이전에 중단된 작업이 있으면 이어받기
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, 'r', encoding='utf-8') as f:
        checkpoint = json.load(f)
    all_results = {int(k): v for k, v in checkpoint['all_results'].items()}
    err_idx = checkpoint['err_idx']
    i = checkpoint['next_i']
    print(f'체크포인트 발견 — {i}번째부터 이어서 시작 (이미 수집: {len(all_results)}건)')
else:
    all_results = dict()
    i = 0
    err_idx = []
    print('새로 시작')

CHECKPOINT_INTERVAL = 100  # 몇 건마다 중간저장할지

for link in naver_news_links[i:]:
    try:
        all_results[i] = dict()

        # 실제 네이버 뉴스 웹페이지로 이동
        driver.get(link)

        # 페이지 로딩 대기
        time.sleep(0.6)

        # # 제목, 본문, 날짜가 로딩될 때까지 대기
        # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'media_end_head_headline')))
        # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'newsct_article')))
        # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME")))
        # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')))

        # 제목 추출하기
        title = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
        title = title[0].text

        # 본문 추출하기
        body = driver.find_elements(By.ID, 'newsct_article')
        body = body[0].text.replace('\n', '')

        # 날짜 추출하기
        pubdate_element = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
        pubdate = pubdate_element[0].get_attribute('data-date-time')

        # 카테고리 추출하기
        category_element = driver.find_elements(By.CSS_SELECTOR, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')
        category = category_element[0].text

        all_results[i]['link']     = link
        all_results[i]['pubdate']  = pubdate
        all_results[i]['category'] = category
        all_results[i]['title']    = title
        all_results[i]['body']     = body

        # 진행 상황 확인용 코드
        print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 진행')
        i += 1

        # 100건마다 중간저장
        if i % CHECKPOINT_INTERVAL == 0:
            with open(checkpoint_path, 'w', encoding='utf-8') as f:
                json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False)
            print(f'  >>> 중간저장 완료 ({i}건)')

        # 봇 탐지 방지를 위해 1.0초에서 2.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(1.0, 2.5))

    # 오류 발생 시
    except:
        print('오류가 발생했습니다.')
        err_idx.append(i)

        print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 진행 - 오류 발생')
        i += 1

        # 봇 탐지 방지를 위해 1.0초에서 2.5초 사이에 랜덤한 시간을 기다림
        time.sleep(random.uniform(1.0, 2.5))

# 완료 후 체크포인트 삭제
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print('체크포인트 삭제 완료')

print(len(all_results))
print(err_idx)

[1 / 5351] 	 0.02% 진행
[2 / 5351] 	 0.04% 진행
[3 / 5351] 	 0.06% 진행
[4 / 5351] 	 0.07% 진행
[5 / 5351] 	 0.09% 진행
[6 / 5351] 	 0.11% 진행
[7 / 5351] 	 0.13% 진행
[8 / 5351] 	 0.15% 진행
[9 / 5351] 	 0.17% 진행
[10 / 5351] 	 0.19% 진행
[11 / 5351] 	 0.21% 진행
[12 / 5351] 	 0.22% 진행
[13 / 5351] 	 0.24% 진행
[14 / 5351] 	 0.26% 진행
[15 / 5351] 	 0.28% 진행
[16 / 5351] 	 0.30% 진행
[17 / 5351] 	 0.32% 진행
[18 / 5351] 	 0.34% 진행
[19 / 5351] 	 0.36% 진행
[20 / 5351] 	 0.37% 진행
[21 / 5351] 	 0.39% 진행
[22 / 5351] 	 0.41% 진행
[23 / 5351] 	 0.43% 진행
[24 / 5351] 	 0.45% 진행
[25 / 5351] 	 0.47% 진행
[26 / 5351] 	 0.49% 진행
[27 / 5351] 	 0.50% 진행
[28 / 5351] 	 0.52% 진행
[29 / 5351] 	 0.54% 진행
[30 / 5351] 	 0.56% 진행
[31 / 5351] 	 0.58% 진행
[32 / 5351] 	 0.60% 진행
[33 / 5351] 	 0.62% 진행
[34 / 5351] 	 0.64% 진행
[35 / 5351] 	 0.65% 진행
[36 / 5351] 	 0.67% 진행
[37 / 5351] 	 0.69% 진행
[38 / 5351] 	 0.71% 진행
[39 / 5351] 	 0.73% 진행
[40 / 5351] 	 0.75% 진행
[41 / 5351] 	 0.77% 진행
[42 / 5351] 	 0.78% 진행
[43 / 5351] 	 0.80% 진행
[44 / 5351] 	 0.82% 

In [ ]:
# err_idx가 빈 리스트가 아닐 때만 자동 재시도
# 전체 주석 설정 및 해제하고 싶으면 Ctrl+A로 전체 선택 후 Ctrl+/ 입력

if len(err_idx) != 0:
    print(f'\n오류 {len(err_idx)}건 재시도 시작...')
    re_err_idx = []

    for i in err_idx:
        try:
            link = naver_news_links[i]
            all_results[i] = dict()

            # 실제 네이버 뉴스 웹페이지로 이동
            driver.get(link)

            # 페이지 로딩 대기
            time.sleep(0.6)

            # # 제목, 본문, 날짜가 로딩될 때까지 대기
            # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'media_end_head_headline')))
            # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'newsct_article')))
            # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME")))

            # 제목 추출하기
            title = driver.find_elements(By.CLASS_NAME, 'media_end_head_headline')
            title = title[0].text

            # 본문 추출하기
            body = driver.find_elements(By.ID, 'newsct_article')
            body = body[0].text.replace('\n', '')

            # 날짜 추출하기
            pubdate_element = driver.find_elements(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
            pubdate = pubdate_element[0].get_attribute('data-date-time')

            # 카테고리 추출하기
            category_element = driver.find_elements(By.CSS_SELECTOR, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')
            category = category_element[0].text if category_element else ''

            all_results[i]['link']     = link
            all_results[i]['pubdate']  = pubdate
            all_results[i]['category'] = category
            all_results[i]['title']    = title
            all_results[i]['body']     = body

            # 진행 상황 확인용 코드
            print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 재시도 성공')

            # 봇 탐지 방지를 위해 1.0초에서 2.5초 사이에 랜덤한 시간을 기다림
            time.sleep(random.uniform(1.0, 2.5))

        # 오류 발생 시
        except:
            re_err_idx.append(i)
            print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% 재시도 실패')
            time.sleep(random.uniform(1.0, 2.5))

    print(f'\n재시도 완료 — 성공: {len(err_idx) - len(re_err_idx)}건 / 재실패: {len(re_err_idx)}건')
    if re_err_idx:
        print(f'재실패 인덱스: {re_err_idx}')
else:
    print('오류 없음 — 재시도 불필요')

오류 없음 — 재시도 불필요


In [ ]:
# 수집한 정보들을 dataframe으로 변환
df = pd.DataFrame(all_results).T
df

,link,pubdate,category,title,body
0,https://n.news.naver.com/mnews/article/629/000...,2025-04-28 11:29:17,사회,유심 해킹 안내문 확인하는 SKT 고객 [포토],유심 해킹 사고가 발생한 SK텔레콤이 전 고객을 대상으로 유심 무상 교체를 실시한 ...
1,https://n.news.naver.com/mnews/article/052/000...,2025-04-27 22:19:52,경제,"과기정통부 ""SKT 해킹 추가 피해방지대책 협의""",과학기술정보통신부가 SK텔레콤 유심 해킹 사고와 관련한 한덕수 대통령 권한대행의 긴...
2,https://n.news.naver.com/mnews/article/030/000...,2025-04-22 11:13:13,IT/과학,SKT 가입자 유심 정보 유출…정부 비상대응 돌입,SK텔레콤 T타워SK텔레콤 고객의 유심(USIM·가입자식별모듈) 정보 일부가 악성코...
3,https://n.news.naver.com/mnews/article/138/000...,2025-04-30 17:41:06,IT/과학,"유상임 장관 “SKT 해지 위약금 면제, 관련 법률 검토 진행중”",30일 서울 여의도 국회에서 열린 법제사법위원회 전체회의에서 유상임 과학기술정보통신...
4,https://n.news.naver.com/mnews/article/056/001...,2025-04-28 16:42:58,사회,"[사사건건] 100% 피해 보상한다는데, 사실일까?","[사사건건] 100% 피해 보상한다는데, 사실일까?KBS뉴스재생21416:28■ 방..."


In [ ]:
# 수집한 기사들 중 중복인 경우 이를 제거
df_no_duplicates = df.drop_duplicates().reset_index(drop=True)
df_no_duplicates

,link,pubdate,category,title,body
0,https://n.news.naver.com/mnews/article/629/000...,2025-04-28 11:29:17,사회,유심 해킹 안내문 확인하는 SKT 고객 [포토],유심 해킹 사고가 발생한 SK텔레콤이 전 고객을 대상으로 유심 무상 교체를 실시한 ...
1,https://n.news.naver.com/mnews/article/052/000...,2025-04-27 22:19:52,경제,"과기정통부 ""SKT 해킹 추가 피해방지대책 협의""",과학기술정보통신부가 SK텔레콤 유심 해킹 사고와 관련한 한덕수 대통령 권한대행의 긴...
2,https://n.news.naver.com/mnews/article/030/000...,2025-04-22 11:13:13,IT/과학,SKT 가입자 유심 정보 유출…정부 비상대응 돌입,SK텔레콤 T타워SK텔레콤 고객의 유심(USIM·가입자식별모듈) 정보 일부가 악성코...
3,https://n.news.naver.com/mnews/article/138/000...,2025-04-30 17:41:06,IT/과학,"유상임 장관 “SKT 해지 위약금 면제, 관련 법률 검토 진행중”",30일 서울 여의도 국회에서 열린 법제사법위원회 전체회의에서 유상임 과학기술정보통신...
4,https://n.news.naver.com/mnews/article/056/001...,2025-04-28 16:42:58,사회,"[사사건건] 100% 피해 보상한다는데, 사실일까?","[사사건건] 100% 피해 보상한다는데, 사실일까?KBS뉴스재생21416:28■ 방..."


In [ ]:
# 오래된 순부터 수집했으나 혹시 모를 상황을 방지하기 위해 pubdate를 datetime으로 변환 후 정렬
df_no_duplicates['pubdate'] = pd.to_datetime(df_no_duplicates['pubdate'])
df_sorted = df_no_duplicates.sort_values(by='pubdate')
df_sorted

,link,pubdate,category,title,body
2,https://n.news.naver.com/mnews/article/030/000...,2025-04-22 11:13:13,IT/과학,SKT 가입자 유심 정보 유출…정부 비상대응 돌입,SK텔레콤 T타워SK텔레콤 고객의 유심(USIM·가입자식별모듈) 정보 일부가 악성코...
1,https://n.news.naver.com/mnews/article/052/000...,2025-04-27 22:19:52,경제,"과기정통부 ""SKT 해킹 추가 피해방지대책 협의""",과학기술정보통신부가 SK텔레콤 유심 해킹 사고와 관련한 한덕수 대통령 권한대행의 긴...
0,https://n.news.naver.com/mnews/article/629/000...,2025-04-28 11:29:17,사회,유심 해킹 안내문 확인하는 SKT 고객 [포토],유심 해킹 사고가 발생한 SK텔레콤이 전 고객을 대상으로 유심 무상 교체를 실시한 ...
4,https://n.news.naver.com/mnews/article/056/001...,2025-04-28 16:42:58,사회,"[사사건건] 100% 피해 보상한다는데, 사실일까?","[사사건건] 100% 피해 보상한다는데, 사실일까?KBS뉴스재생21416:28■ 방..."
3,https://n.news.naver.com/mnews/article/138/000...,2025-04-30 17:41:06,IT/과학,"유상임 장관 “SKT 해지 위약금 면제, 관련 법률 검토 진행중”",30일 서울 여의도 국회에서 열린 법제사법위원회 전체회의에서 유상임 과학기술정보통신...


In [ ]:
# 수집한 정보들을 csv로 저장 (이때 인코딩 형식은 utf8)
# 저장할 하위폴더 지정 (현재 작업 폴더 아래)
subfolder = 'data'  # 필요에 따라 변경. 빈 문자열로 하면 현재 폴더에 저장
save_dir = os.path.join(os.getcwd(), subfolder) if subfolder else os.getcwd()
os.makedirs(save_dir, exist_ok=True)

file_name = f"{query}_{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}.csv"
save_path = os.path.join(save_dir, file_name)

df_sorted.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {save_path}')

저장 완료: c:\Users\carol\Desktop\대학\KMU\4학년 1학기\텍스트데이터분석\텍데분 프로젝트\data\SK텔레콤_250401_250430.csv


In [ ]:
# 브라우저 창 닫기
driver.quit()